# 🤖 Machine Learning — Sessions 5 & 7
## SVM + Ensemble Methods (Bagging & Boosting)

---

## 📚 فهرس المحتوى
1. Support Vector Machines (SVM)
2. XGBoost — Classification
3. XGBoost — Regression
4. XGBoost — GridSearchCV
5. Bagging — Random Forest
6. Boosting — AdaBoost
7. Boosting — Gradient Boosting
8. مقارنة شاملة بين الخوارزميات

## 📦 تثبيت المكتبات

In [ ]:
!pip install xgboost

## 📥 Imports المشتركة

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.datasets import (
    load_iris,
    load_breast_cancer,
    fetch_california_housing,
    make_blobs,
    make_moons
)

import xgboost as xgb
from xgboost import XGBClassifier, XGBRegressor

print('✅ XGBoost version:', xgb.__version__)
print('✅ All libraries imported successfully!')

---
# 1. 🎯 Support Vector Machines (SVM)

**المفهوم:**
- SVM بيدور على أفضل **Hyperplane** يفصل الفئتين بأكبر **Margin** ممكن
- **Support Vectors** = النقاط الأقرب للـ Hyperplane
- **Kernel Trick** = تحويل البيانات لـ Space أعلى أبعاداً لفصل البيانات غير الخطية

### 1.1 SVM بـ Linear Kernel على Iris Dataset

In [ ]:
from sklearn.svm import SVC

# تحميل البيانات
iris = load_iris()
X = iris.data
y = iris.target

print('شكل البيانات:', X.shape)
print('الفئات:', iris.target_names)

In [ ]:
# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

In [ ]:
# بناء وتدريب الموديل
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)

# التنبؤ
y_pred = svm_model.predict(X_test)

# التقييم
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# رسم الـ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=iris.target_names,
    yticklabels=iris.target_names
)
plt.title('SVM (Linear Kernel) — Confusion Matrix', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

### 1.2 مقارنة الـ Kernels المختلفة

In [ ]:
# مقارنة أداء الـ Kernels المختلفة
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
kernel_scores = {}

print('📊 مقارنة أداء الـ Kernels المختلفة:')
print('-' * 40)

for kernel in kernels:
    print(f'\n========== {kernel.upper()} Kernel ==========')

    model = SVC(kernel=kernel)
    model.fit(X_train, y_train)
    y_pred_k = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred_k)
    kernel_scores[kernel] = acc
    print('Accuracy:', acc)

In [ ]:
# رسم مقارنة الـ Kernels
plt.figure(figsize=(8, 5))
bars = plt.bar(
    kernel_scores.keys(),
    kernel_scores.values(),
    color=['#4ECDC4', '#FF6B6B', '#45B7D1', '#96CEB4'],
    edgecolor='white',
    linewidth=1.5
)

for bar, val in zip(bars, kernel_scores.values()):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{val:.3f}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

plt.title('SVM — مقارنة أداء الـ Kernels المختلفة', fontsize=13)
plt.ylabel('Accuracy')
plt.xlabel('Kernel Type')
plt.ylim(0.8, 1.05)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 1.3 SVM مع Feature Scaling (Pipeline)

In [ ]:
from sklearn.pipeline import Pipeline

# Pipeline: Scaling → SVM
# مهم جداً: SVM حساس لـ Feature Scaling!
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

svm_pipeline.fit(X_train, y_train)
y_pred_pipeline = svm_pipeline.predict(X_test)

print('✅ Accuracy with Scaling + RBF Kernel:', accuracy_score(y_test, y_pred_pipeline))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_pipeline, target_names=iris.target_names))

---
# 2. ⚡ XGBoost — Classification

**المفهوم:**
- XGBoost = Gradient Boosting محسّن مع Regularization + Parallelization
- أفضل خوارزمية في مسابقات Kaggle 🏆
- بيتعلم بشكل تسلسلي: كل شجرة بتصلح أخطاء اللي قبلها

In [ ]:
# تحميل Breast Cancer Dataset
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print('شكل البيانات:', X.shape)
print('\nأول 5 صفوف:')
print(X.head())
print('\nالفئات:', data.target_names)
print('توزيع الفئات:', dict(zip(*np.unique(y, return_counts=True))))

In [ ]:
# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y    # يحافظ على نسب الفئات في التقسيم
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

In [ ]:
# بناء وتدريب XGBClassifier
model = XGBClassifier(
    n_estimators=100,          # عدد الأشجار
    max_depth=3,               # عمق كل شجرة
    max_colsample_bytree=0.8,  # نسبة الـ Features في كل شجرة
    learning_rate=0.1,         # معدل التعلم (Shrinkage)
    random_state=42
)

model.fit(X_train, y_train)
print('✅ Model trained successfully!')

In [ ]:
# التنبؤ والتقييم
y_pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# رسم الـ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=data.target_names,
    yticklabels=data.target_names
)
plt.title('XGBoost Classifier — Confusion Matrix', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feat_importance = pd.Series(
    model.feature_importances_,
    index=data.feature_names
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_importance.head(10).plot(kind='bar', color='#45B7D1', edgecolor='white')
plt.title('XGBoost — Top 10 Feature Importances', fontsize=13)
plt.ylabel('Importance')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
# 3. 🏠 XGBoost — Regression

**Dataset:** California Housing — التنبؤ بأسعار المنازل

In [ ]:
# تحميل البيانات
housing = fetch_california_housing()

X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

print('شكل البيانات:', X.shape)
print('\nأول 5 صفوف:')
print(X.head())
print('\nأول 5 قيم Target (سعر المنزل بـ $100,000):', y[:5])

In [ ]:
# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

In [ ]:
# بناء وتدريب XGBRegressor
reg_model = XGBRegressor(
    random_state=42,
    objective='reg:squarederror',  # Loss Function للـ Regression
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    colsample_bytree=0.8,  # نسبة الـ Features (columns) في كل شجرة
    subsample=0.8          # نسبة الـ Rows في كل شجرة
)

reg_model.fit(X_train, y_train)
print('✅ Regression model trained successfully!')

In [ ]:
# التنبؤ والتقييم
y_pred = reg_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('📊 Regression Metrics:')
print(f'   MAE  (Mean Absolute Error):  {mae:.4f}')   # متوسط الخطأ المطلق
print(f'   MSE  (Mean Squared Error):   {mse:.4f}')   # متوسط مربع الخطأ
print(f'   RMSE (Root MSE):             {rmse:.4f}')  # نفس وحدة الـ Target
print(f'   R²   (R-Squared):            {r2:.4f}')    # نسبة التفسير (1 = مثالي)

In [ ]:
# رسم Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='#4ECDC4', s=20)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--', lw=2, label='Perfect Prediction'
)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'XGBoost Regressor — Actual vs Predicted\nR² = {r2:.4f}', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# 4. 🔧 XGBoost — GridSearchCV (ضبط الـ Hyperparameters)

**المفهوم:** GridSearchCV بيجرب كل التوليفات الممكنة من الـ Hyperparameters
ويختار الأفضل باستخدام Cross Validation

In [ ]:
# إعادة تحميل Breast Cancer للـ Classification
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# تعريف الـ Grid
model = XGBClassifier(random_state=42)

param_grid = {
    'n_estimators':     [100, 200, 300],
    'learning_rate':    [0.01, 0.1, 0.2],
    'max_depth':        [3, 5],
    'subsample':        [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

print('إجمالي التوليفات:', 3*3*2*3*3, 'توليفة')
print('مع 5-Fold CV:', 3*3*2*3*3*5, 'موديل!')

In [ ]:
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,        # 5-Fold Cross Validation
    n_jobs=-1,   # استخدم كل الـ CPU Cores
    verbose=1
)

grid_search.fit(X_train, y_train)

print('\n✅ Best Parameters:')
print(grid_search.best_params_)

print('\n✅ Best CV Score:')
print(grid_search.best_score_)

In [ ]:
# تقييم الـ Best Model
y_pred = grid_search.predict(X_test)

print('✅ Test Accuracy (Best Model):', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

---
# 5. 🎒 Bagging — Random Forest

**المفهوم:**
- Bagging = Bootstrap Aggregating
- بيدرب موديلات مستقلة على Subsets مختلفة (Sampling with Replacement)
- Random Forest = Bagging + Decision Trees + Feature Randomness
- بيقلل الـ **Variance** ويمنع الـ Overfitting

In [ ]:
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# إعادة تحميل Iris
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# Decision Tree وحدها (للمقارنة)
# ==============================
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt.predict(X_test))
print(f'Decision Tree Accuracy:    {dt_acc:.4f}')

# ==============================
# Bagging Classifier
# ==============================
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,        # عدد الموديلات
    max_samples=0.8,         # نسبة الـ Samples في كل Subset
    bootstrap=True,          # Sampling with Replacement
    random_state=42,
    n_jobs=-1
)
bagging.fit(X_train, y_train)
bag_acc = accuracy_score(y_test, bagging.predict(X_test))
print(f'Bagging Classifier Accuracy: {bag_acc:.4f}')

# ==============================
# Random Forest
# ==============================
rf = RandomForestClassifier(
    n_estimators=100,   # عدد الأشجار في الغابة
    max_depth=None,     # الأشجار بتكبر لحد ما
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))
print(f'Random Forest Accuracy:    {rf_acc:.4f}')

In [ ]:
# تأثير عدد الأشجار (n_estimators) على الأداء
n_estimators_range = [1, 5, 10, 25, 50, 100, 200, 500]
rf_scores = []

for n in n_estimators_range:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_temp.fit(X_train, y_train)
    rf_scores.append(accuracy_score(y_test, rf_temp.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(n_estimators_range, rf_scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('عدد الأشجار (n_estimators)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Random Forest — تأثير عدد الأشجار على الأداء', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('💡 بعد نقطة معينة الأداء بيثبت ومش بيتحسن كتير')

In [ ]:
# Hard Voting vs Soft Voting
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Hard Voting
hard_voting = VotingClassifier(
    estimators=[
        ('dt',  DecisionTreeClassifier(random_state=42)),
        ('svm', SVC(kernel='linear', probability=True, random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ],
    voting='hard'
)

# Soft Voting
soft_voting = VotingClassifier(
    estimators=[
        ('dt',  DecisionTreeClassifier(random_state=42)),
        ('svm', SVC(kernel='linear', probability=True, random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ],
    voting='soft'
)

hard_voting.fit(X_train, y_train)
soft_voting.fit(X_train, y_train)

print('📊 Voting Comparison:')
print(f'   Hard Voting Accuracy: {accuracy_score(y_test, hard_voting.predict(X_test)):.4f}')
print(f'   Soft Voting Accuracy: {accuracy_score(y_test, soft_voting.predict(X_test)):.4f}')

---
# 6. 🚀 Boosting — AdaBoost

**المفهوم:**
- AdaBoost = Adaptive Boosting
- بيدرب Weak Learners (Decision Stumps) بشكل تسلسلي
- النقاط المُصنَّفة غلط → بتاخد وزن أعلى في الجولة الجاية
- بيقلل الـ **Bias** عن طريق التركيز على الحالات الصعبة

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

# إعادة تحميل Breast Cancer
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# AdaBoost مع Decision Stump (الأساسي)
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision Stump
    n_estimators=100,       # عدد الجولات
    learning_rate=1.0,      # معدل تأثير كل Learner
    random_state=42
)

ada_model.fit(X_train, y_train)
y_pred = ada_model.predict(X_test)

print('✅ AdaBoost Results:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# تأثير عدد الـ Estimators على أداء AdaBoost
n_est_range = [1, 5, 10, 25, 50, 100, 150, 200]
ada_train_scores = []
ada_test_scores  = []

for n in n_est_range:
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n,
        random_state=42
    )
    ada.fit(X_train, y_train)
    ada_train_scores.append(accuracy_score(y_train, ada.predict(X_train)))
    ada_test_scores.append(accuracy_score(y_test, ada.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(n_est_range, ada_train_scores, 'b-o', label='Train Accuracy', linewidth=2)
plt.plot(n_est_range, ada_test_scores,  'r-o', label='Test Accuracy',  linewidth=2)
plt.xlabel('عدد الـ Estimators', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('AdaBoost — Train vs Test Accuracy', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# 7. 📈 Boosting — Gradient Boosting

**المفهوم:**
- بدل تحديث الأوزان (زي AdaBoost)، بيتدرب كل Learner على الـ **Residuals** (الأخطاء)
- بيستخدم **Gradient Descent** لتقليل الـ Loss Function
- XGBoost هو نسخة محسّنة منه

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(
    n_estimators=100,    # عدد الأشجار
    max_depth=3,         # عمق كل شجرة
    learning_rate=0.1,   # معدل التعلم
    subsample=0.8,       # نسبة الـ Samples في كل شجرة
    random_state=42
)

gb_model.fit(X_train, y_train)
y_pred = gb_model.predict(X_test)

print('✅ Gradient Boosting Results:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# تأثير الـ Learning Rate على الأداء
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
lr_scores = []

for lr in learning_rates:
    gb = GradientBoostingClassifier(
        n_estimators=100, max_depth=3,
        learning_rate=lr, random_state=42
    )
    gb.fit(X_train, y_train)
    lr_scores.append(accuracy_score(y_test, gb.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.semilogx(learning_rates, lr_scores, 'go-', linewidth=2, markersize=8)
plt.xlabel('Learning Rate (log scale)', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Gradient Boosting — تأثير الـ Learning Rate', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_lr = learning_rates[np.argmax(lr_scores)]
print(f'✅ أفضل Learning Rate: {best_lr}')

---
# 8. ⚖️ مقارنة شاملة بين الخوارزميات

In [ ]:
import time
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.svm import SVC

# إعداد البيانات
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# تعريف الخوارزميات
models = {
    'SVM (RBF)':           SVC(kernel='rbf', random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost':             AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':              XGBClassifier(n_estimators=100, random_state=42)
}

# تدريب وتقييم كل موديل
results = {}

print('📊 مقارنة شاملة بين الخوارزميات:')
print('=' * 60)
print(f'{"Algorithm":<25} {"Accuracy":>10} {"Time (s)":>10}')
print('-' * 60)

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start

    acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = {'accuracy': acc, 'time': elapsed}

    print(f'{name:<25} {acc:>10.4f} {elapsed:>10.4f}')

print('=' * 60)

In [ ]:
# رسم المقارنة
names  = list(results.keys())
accs   = [results[n]['accuracy'] for n in names]
times  = [results[n]['time'] for n in names]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
bars = axes[0].bar(names, accs, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, accs):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.001,
        f'{val:.4f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
axes[0].set_title('مقارنة الـ Accuracy', fontsize=13)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.9, 1.02)
axes[0].set_xticklabels(names, rotation=20, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Training Time
bars2 = axes[1].bar(names, times, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars2, times):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.001,
        f'{val:.3f}s',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
axes[1].set_title('مقارنة وقت التدريب', fontsize=13)
axes[1].set_ylabel('Time (seconds)')
axes[1].set_xticklabels(names, rotation=20, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('مقارنة شاملة: SVM vs Bagging vs Boosting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cross Validation مقارنة (أكثر دقة)
print('📊 5-Fold Cross Validation Comparison:')
print('=' * 50)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name:<25}: {scores.mean():.4f} ± {scores.std():.4f}')

print('=' * 50)

In [ ]:
# Boxplot للـ Cross Validation Scores
plt.figure(figsize=(10, 6))
plt.boxplot(
    [cv_results[n] for n in names],
    labels=names,
    patch_artist=True,
    boxprops=dict(facecolor='#4ECDC4', color='#333'),
    medianprops=dict(color='#FF6B6B', linewidth=2),
    whiskerprops=dict(color='#333'),
    capprops=dict(color='#333')
)
plt.title('Cross Validation Scores — مقارنة الاستقرار', fontsize=13)
plt.ylabel('Accuracy')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\n🔑 الخلاصة:')
print('   ✦ SVM        = مناسب للـ High-Dimensional Spaces')
print('   ✦ Random Forest = سريع + مستقر + يقلل Overfitting')
print('   ✦ AdaBoost   = بسيط + بيانات نظيفة')
print('   ✦ Grad Boost = قوي + بيانات Tabular')
print('   ✦ XGBoost    = الأفضل غالباً ✅ (سريع + Regularization)')